# 2주차 실습 (06) - 확률과 분포 from scratch

PMF/PDF, 기댓값/분산, **수치 안정 softmax/log-softmax**, 중심극한정리(CLT)를
표준 라이브러리만으로 구현하고 확인한다.

## Step 1. PMF / PDF

In [ ]:
import math
import random


def factorial(n):
    r = 1
    for i in range(2, n + 1):
        r *= i
    return r


def bernoulli_pmf(k, p):
    return p if k == 1 else (1 - p)


def poisson_pmf(k, lam):
    return (lam ** k) * math.exp(-lam) / factorial(k)


def normal_pdf(x, mu, sigma):
    coeff = 1.0 / (sigma * math.sqrt(2 * math.pi))
    return coeff * math.exp(-0.5 * ((x - mu) / sigma) ** 2)


print(f"Poisson(k=2, λ=3) = {poisson_pmf(2, 3):.4f}")
print(f"Normal(0, μ=0, σ=1) = {normal_pdf(0, 0, 1):.4f}")   # 0.3989 = 1/√2π

## Step 2. 기댓값과 분산

`E[X]=Σx·P(x)`, `Var(X)=E[(X-μ)²]`. 공정한 주사위는 `E[X]=3.5`, `Var=2.9167`.

In [ ]:
def expected_value(values, probs):
    return sum(v * p for v, p in zip(values, probs))


def variance(values, probs):
    mu = expected_value(values, probs)
    return sum(p * (v - mu) ** 2 for v, p in zip(values, probs))


vals, probs = [1, 2, 3, 4, 5, 6], [1 / 6] * 6
mu, var = expected_value(vals, probs), variance(vals, probs)
print(f"E[X]={mu:.4f}  Var={var:.4f}  SD={var ** 0.5:.4f}")

## Step 3. softmax / log-softmax (수치 안정)

지수 전에 `max(z)`를 빼서 오버플로 방지. 결과는 동일하지만 `exp(102)` 같은 폭주를 막는다.

In [ ]:
def softmax(logits):
    m = max(logits)
    exps = [math.exp(z - m) for z in logits]
    s = sum(exps)
    return [e / s for e in exps]


def log_softmax(logits):
    m = max(logits)
    log_sum_exp = m + math.log(sum(math.exp(z - m) for z in logits))
    return [z - log_sum_exp for z in logits]


def cross_entropy_loss(logits, target_index):
    return -log_softmax(logits)[target_index]


print(f"softmax([100,101,102]) = {[round(x, 4) for x in softmax([100, 101, 102])]}")
print(f"CE loss (정답=3) = {cross_entropy_loss([2.0, 0.5, -1.0, 3.0, 0.1], 3):.4f}")

## Step 4. 중심극한정리 (CLT)

원분포가 균등(주사위)이어도, 여러 개를 평균 내면 정규분포에 가까워지고 표준편차가 줄어든다.

In [ ]:
def demonstrate_clt(dist_fn, n_samples, n_averages):
    return [sum(dist_fn() for _ in range(n_samples)) / n_samples
            for _ in range(n_averages)]


random.seed(42)
die = lambda: random.randint(1, 6)
for n in (1, 2, 30):
    avgs = demonstrate_clt(die, n_samples=n, n_averages=5000)
    m = sum(avgs) / len(avgs)
    sd = (sum((a - m) ** 2 for a in avgs) / len(avgs)) ** 0.5
    print(f"주사위 {n:2d}개 평균:  mean={m:.3f}  sd={sd:.3f}  (n↑ 이면 sd↓)")

## (선택) Step 5. 정규분포 PDF 시각화

matplotlib 이 있으면 종형 곡선을 그린다.

In [ ]:
try:
    import matplotlib.pyplot as plt

    xs = [(i - 400) / 100 for i in range(801)]   # -4 ~ 4
    ys = [normal_pdf(x, 0, 1) for x in xs]
    plt.figure(figsize=(6, 3))
    plt.plot(xs, ys)
    plt.title("Standard Normal PDF")
    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib 미설치 - pip install matplotlib")

## 정리

- 분류 출력 = PMF, 연속 잠재공간 = PDF. softmax + log 확률이 크로스 엔트로피의 재료.
- 정규분포가 어디에나 나오는 이유 = CLT.
- 개념 노트: [[📊 확률과 분포]] (Obsidian LLM_Wiki)